# 03 · Evaluation (FA / CER / F1) — đa model VLM

Giai đoạn 3: đánh giá trên tập **Test** (chưa từng thấy lúc train).

Đổi `MODEL_KEY` ở cell đầu rồi **chạy lại toàn bộ notebook cho từng model** (`qwen`,
`internvl`, `llama_vision`). Mỗi lần chạy sinh 2 report — fine-tuned và zero-shot —
đặt tên theo `model_key` để `scripts/compare_models.py` gom nhóm được.

Mọi run dùng **cùng tập test, cùng prompt/schema, cùng `metrics.evaluate`, cùng greedy
decoding**. Dự đoán từng ảnh cũng được ghi ra `*_preds.jsonl` để notebook 05 tính
bootstrap CI và McNemar test.

**Báo cáo lưu vào Google Drive.**

## 1. Mount Drive + đường dẫn

In [ ]:
# ═══ CẤU HÌNH — ĐỔI MODEL_KEY RỒI CHẠY LẠI CHO TỪNG MODEL ════════════════
#   'qwen' | 'internvl' | 'llama_vision'
MODEL_KEY = 'qwen'
SIDE = 'Back'
# ═════════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import sys
from pathlib import Path

DATA_DRIVE = Path('/content/drive/MyDrive/cccd_project/Data')
REPO_DRIVE = DATA_DRIVE / 'label_CCCD'
sys.path.insert(0, str(REPO_DRIVE))
from src.models.vlm_registry import resolve, resolve_model_dir

spec        = resolve(MODEL_KEY)
MODEL_ID    = spec.model_id

MODELS_DIR = DATA_DRIVE / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Nơi đặt snapshot base model do resolve_model_dir quyết định, KHÔNG hardcode:
# Llama-3.2-11B-Vision nặng ~21GB, không vừa Google Drive free (15GB) → tải nửa
# chừng rồi chết, để lại thư mục thiếu shard. Hàm này ưu tiên Drive (bền qua các
# lần Colab ngắt), nhưng rơi về SSD /content/models khi Drive không đủ chỗ, và
# luôn dùng lại nơi nào ĐÃ có snapshot đủ file.
MODEL_LOCAL = resolve_model_dir(spec, MODELS_DIR)
SSD_NOTE = '' if str(MODEL_LOCAL).startswith('/content/drive') else \
    '   ⚠ SSD tạm — mất khi ngắt phiên, notebook sau phải tải lại'

TEST_JSONL = DATA_DRIVE / 'dataset' / SIDE / 'test.jsonl'
IMAGE_ROOT = DATA_DRIVE / SIDE

# Khớp quy ước đặt tên của notebook 02.
CKPT_DIR = DATA_DRIVE / 'checkpoints' / f'{MODEL_KEY}-cccd-lora-{SIDE.lower()}'

# [ĐÃ SỬA] Report tách theo MẶT THẺ. compare_models.py gom nhóm theo
# model_key × mode và KHÔNG phân biệt Front/Back — để chung một thư mục thì
# eval_qwen_ft_front và eval_qwen_ft_back bị coi là trùng nhau, script chỉ giữ
# file mới nhất (log 'Trùng qwen/fine_tuned') và bảng so sánh lẫn hai mặt.
RESULT_DIR = DATA_DRIVE / 'result' / SIDE.lower()
RESULT_DIR.mkdir(parents=True, exist_ok=True)
# Tên report có model_key để compare_models.py gom nhóm đúng.
REPORT_FT       = RESULT_DIR / f'eval_{MODEL_KEY}_ft_{SIDE.lower()}.json'
REPORT_ZEROSHOT = RESULT_DIR / f'eval_{MODEL_KEY}_zeroshot_{SIDE.lower()}.json'

print(f'--- ĐÁNH GIÁ: {MODEL_ID}  [{spec.key}] ---')
print('✅ Test JSONL   :', TEST_JSONL)
print('📂 Base Model   :', MODEL_LOCAL, SSD_NOTE)
print('📂 Checkpoint   :', CKPT_DIR, '' if CKPT_DIR.exists() else '  ⚠ CHƯA CÓ — chạy notebook 02 trước')
print('💾 Report FT    :', REPORT_FT)
print('💾 Report ZS    :', REPORT_ZEROSHOT)
if spec.gated:
    print('\n⚠ Model gated — cần HF_TOKEN trong Colab Secrets nếu base chưa có sẵn local.')

## 2. Cài thư viện (bản ổn định, KHỚP notebook 02 lúc train)

In [ ]:
%cd /content/drive/MyDrive/cccd_project/Data/label_CCCD

# [ĐÃ SỬA] PIN cùng phiên bản transformers với lúc train (không cài bản 'git' dev).
# Trước đây eval cài bản dev → lệch chat-template / cách bung image-token so với
# lúc train → adapter chạy sai, sinh chuỗi rác.
# Version tối thiểu lấy từ registry theo MODEL_KEY đang chọn.
!pip -q install 'transformers>={spec.min_transformers}' qwen-vl-utils accelerate peft bitsandbytes Pillow tqdm

import transformers
from src.models.vlm_registry import check_available
ok, reason = check_available(spec)
print(f'✅ transformers {transformers.__version__} | {spec.key} khả dụng: {ok} ({reason})')

In [ ]:
# --- 2c. KIỂM TRA BẢN CODE TRÊN DRIVE ĐÃ CÓ BẢN VÁ PARSER CHƯA ---
# Notebook này chạy THẲNG repo trên Drive (%cd ở cell trên), không copy sang SSD như
# notebook 02. Sửa code ở máy mà quên đồng bộ lên Drive là các cell dưới vẫn chạy bằng
# bản cũ, ra đúng con số cũ, và KHÔNG có lỗi nào báo ra. Cell này chặn đúng chỗ đó.
#
# Chạy được với cả hai bố cục repo:
#   cũ  : scripts/*.py và src/ là mã nguồn thật
#   mới : model/scripts/*.py là thật, scripts/ và src/ ở gốc chỉ là shim
from pathlib import Path

from src.utils.metrics import safe_parse


def _find(name: str):
    """Tìm script thật, ưu tiên bố cục mới rồi mới đến bố cục cũ."""
    for candidate in (Path('model/scripts') / name, Path('scripts') / name):
        if candidate.exists():
            return candidate
    return None


# 1) Parser: kiểm tra bằng HÀNH VI, không phụ thuộc file nằm ở đâu.
_fenced = '```json\n{"so_cccd": "012345678901"}\n```'
_lead = 'Thông tin trên thẻ:\n{"so_cccd": "012345678901"}'
_ok_parser = bool(safe_parse(_fenced)) and bool(safe_parse(_lead))

# 2) evaluate.py phải ghi trường `raw` — nếu không, mọi ảnh parse hỏng sẽ mất chuỗi
#    thô và lần sau đổi parser lại phải chạy GPU cho cả tập test.
_eval_py = _find('evaluate.py')
_ok_raw = bool(_eval_py) and '"raw": raw' in _eval_py.read_text(encoding='utf-8')

# 3) rescore.py phải gọi được qua đường `scripts/` (bố cục mới có shim ở đó).
_rescore_py = _find('rescore.py')
RESCORE_PY = Path('scripts/rescore.py') if Path('scripts/rescore.py').exists() else _rescore_py
EVALUATE_PY = Path('scripts/evaluate.py') if Path('scripts/evaluate.py').exists() else _eval_py

print(f'safe_parse nới rào/câu dẫn : {"✓" if _ok_parser else "✗ BẢN CŨ"}')
print(f'evaluate.py ghi trường raw : {"✓" if _ok_raw else "✗ BẢN CŨ"}   ({_eval_py})')
print(f'rescore.py                 : {"✓" if RESCORE_PY else "✗ THIẾU"}   ({RESCORE_PY})')

if not (_ok_parser and _ok_raw and RESCORE_PY):
    raise SystemExit(
        'Repo trên Drive chưa có bản vá. Đồng bộ lại rồi chạy lại cell này:\n'
        '  src/utils/metrics.py  (hoặc model/src/...)  — safe_parse nới 3 nhánh parse\n'
        '  scripts/evaluate.py   (hoặc model/scripts/...) — ghi thêm `raw` vào preds\n'
        '  scripts/rescore.py    (hoặc model/scripts/...) — script chấm lại\n\n'
        'Bản cũ chỉ gọi json.loads trần: model bọc đáp án trong ```json hoặc thêm câu\n'
        'dẫn là parse hỏng → mọi trường bị chấm sai, dù đọc thẻ đúng.'
    )
print('\n✅ Repo trên Drive đã đúng bản vá.')

## 2b. Đảm bảo base model có ĐỦ trọng số ở local
Bắt buộc chạy: notebook 02 có thể đã tải model xuống SSD `/content` (khi Drive không đủ chỗ), và SSD mất sau mỗi lần ngắt phiên.


In [ ]:
# --- TẢI BASE MODEL VỀ LOCAL (đủ trọng số mới thôi) ---
# [ĐÃ SỬA] Trước đây guard bằng `if not (MODEL_LOCAL/'config.json').exists()`.
# snapshot_download tải file nhỏ (config, index, tokenizer) TRƯỚC, shard nặng SAU →
# lần tải bị đứt (hết chỗ trên Drive / Colab disconnect) vẫn để lại config.json,
# guard tưởng "đã có sẵn" nên bỏ qua, và lỗi chỉ nổ ra ở tận lúc nạp model:
#     FileNotFoundError: .../model-00001-of-00005.safetensors
# ensure_snapshot() đối chiếu model.safetensors.index.json nên biết thiếu shard nào
# (và bắt được cả shard ghi dở), rồi tải tiếp — snapshot_download resume được.
from src.models.vlm_registry import ensure_snapshot, snapshot_complete

ok, reason = snapshot_complete(MODEL_LOCAL)
print(f'Snapshot hiện có: {"đủ file" if ok else reason}  →  {MODEL_LOCAL}')

# Model gated (Llama-3.2-Vision) phải đăng nhập TRƯỚC khi tải. Đã có sẵn thì không cần.
if spec.gated and not ok:
    from huggingface_hub import login
    try:
        from google.colab import userdata
        login(userdata.get('HF_TOKEN'))
        print('✓ Đã đăng nhập HuggingFace')
    except Exception as exc:
        raise SystemExit(
            f'{spec.model_id} là model gated nhưng chưa đăng nhập được ({exc}).\n'
            f'1) Vào https://huggingface.co/{spec.model_id} accept license\n'
            f'2) Tạo token tại https://huggingface.co/settings/tokens\n'
            f'3) Colab → 🔑 Secrets → thêm HF_TOKEN, bật "Notebook access"'
        )

ensure_snapshot(spec, MODEL_LOCAL)   # raise nếu tải xong mà vẫn thiếu file
print('✓ Base model sẵn sàng:', MODEL_LOCAL)


## 3. Chạy đánh giá (base LOCAL + adapter)

In [ ]:
# --- HÀNG 1/2: FINE-TUNED ---
# evaluate.py tự đọc vlm_meta.json cạnh adapter để biết base model đã train, và
# cảnh báo nếu --base_model truyền vào khác base lúc train.
# Decode bị ép greedy tường minh (do_sample=False, repetition_penalty=1.0) nên
# kết quả tái lập được và so sánh được giữa các model.
# Dự đoán từng ảnh được ghi kèm (*_preds.jsonl) — notebook 05 cần để tính CI/McNemar.
!python scripts/evaluate.py \
    --test_jsonl '{TEST_JSONL}' \
    --base_model '{MODEL_LOCAL}' \
    --adapter_dir '{CKPT_DIR}' \
    --model_key '{MODEL_KEY}' \
    --image_root '{IMAGE_ROOT}' \
    --report_path '{REPORT_FT}'

In [ ]:
# --- HÀNG 2/2: ZERO-SHOT (base model thuần, KHÔNG adapter) ---
# Bắt buộc có cho bảng so sánh: hiệu (fine-tuned − zero-shot) chính là đóng góp
# thuần của QLoRA cho kiến trúc này. Dùng y hệt prompt / tiền xử lý ảnh / tham số
# sinh với hàng fine-tuned; khác biệt DUY NHẤT là có gắn adapter hay không.
!python scripts/evaluate.py \
    --test_jsonl '{TEST_JSONL}' \
    --base_model '{MODEL_LOCAL}' \
    --model_key '{MODEL_KEY}' \
    --zero_shot \
    --image_root '{IMAGE_ROOT}' \
    --report_path '{REPORT_ZEROSHOT}'

## 4. Xem báo cáo + per-field accuracy

In [ ]:
import json
from pathlib import Path

def show(path, title):
    """In một report; trả về dict (None nếu chưa chạy)."""
    if not Path(path).exists():
        print(f'➖ Chưa có: {title}')
        return None
    with open(path, encoding='utf-8') as f:
        r = json.load(f)
    run = r.get('run', {})
    print(f'\n{"="*56}\n{title}   [{run.get("model_key","?")} / {run.get("mode","?")}]\n{"="*56}')
    print(f'  Field Accuracy    : {r["field_accuracy"]:.2%}')
    print(f'  CER               : {r["cer"]:.4f}')
    print(f'  F1 (micro)        : {r["f1"]:.2%}   (P={r["precision"]:.2%} R={r["recall"]:.2%})')
    print(f'  JSON parse rate   : {r.get("json_parse_rate", 0):.2%}   lỗi: {r.get("n_errors", 0)}')
    lat = r.get('latency_ms', {})
    if lat:
        print(f'  Latency p50/p95   : {lat.get("p50")} / {lat.get("p95")} ms  (mean {lat.get("mean")})')
    if r.get('peak_vram_mb'):
        print(f'  Peak VRAM         : {r["peak_vram_mb"]/1024:.2f} GB')
    if run.get('trainable_params_pct') is not None:
        print(f'  Trainable params  : {run["trainable_params_pct"]:.4f}%')
    print(f'  Decoding          : {run.get("decoding","?")}')
    print('\n  Per-field accuracy:')
    for k, v in sorted(r['per_field_accuracy'].items(), key=lambda x: x[1], reverse=True):
        print(f'    {k:25s} {v:.2%}')
    return r

ft = show(REPORT_FT,       f'🏆 FINE-TUNED ({MODEL_KEY})')
zs = show(REPORT_ZEROSHOT, f'🧊 ZERO-SHOT ({MODEL_KEY})')

# Delta — con số quan trọng nhất cho báo cáo
if ft and zs:
    print(f'\n{"="*56}\n📈 ĐÓNG GÓP CỦA QLoRA (fine-tuned − zero-shot)\n{"="*56}')
    print(f'  Δ Field Accuracy : {ft["field_accuracy"] - zs["field_accuracy"]:+.2%}')
    print(f'  Δ CER            : {ft["cer"] - zs["cer"]:+.4f}  (âm là tốt)')
    print(f'  Δ F1             : {ft["f1"] - zs["f1"]:+.2%}')

print('\n👉 Chạy xong cả 3 model thì sang notebook 05 để gộp bảng + kiểm định thống kê.')

## 5. Chấm lại sau khi sửa parser (`safe_parse`)

Chỉ chạy phần này **cho các report đã sinh TRƯỚC bản vá parser**. Report sinh từ bây
giờ trở đi đã có sẵn trường `raw` nên chấm lại chỉ tốn vài giây CPU, không cần GPU.

**Vì sao phải chấm lại.** `metrics.safe_parse` bản cũ chỉ gọi `json.loads` trần: model
nào bọc đáp án trong ```` ```json ```` hoặc thêm câu dẫn là parse hỏng → `pred = {}` →
**mọi trường bị chấm sai** dù đọc thẻ đúng. Đó là lý do InternVL zero-shot có
`json_parse_rate = 0/144` và `F1 = 0.00%` tuyệt đối.

**Phần nào KHÔNG cần chạy lại:**

- **Notebook 02 — không đụng gì.** `safe_parse` chỉ nằm ở đường chấm điểm; loss lúc
  train tính ở mức token, không parse JSON. Adapter đã train là bất biến.
- **Sáu run fine-tuned — số không đổi.** Cả 6 đều có `json_parse_rate = 100%`, tức
  `json.loads` đã thành công; parser mới chỉ *thêm nhánh dự phòng khi `json.loads`
  thất bại* nên cho ra đúng dict cũ. Cell dưới vẫn chạy chúng để **xác nhận** điều đó
  — nếu FA đổi dù chỉ 0.01 điểm là có gì đó sai, phải dừng lại xem.

**Cell dưới làm gì.** Với từng report (FT và zero-shot):

1. `rescore missing` — lọc ra ảnh có `pred` rỗng **và** mất `raw` (bắt buộc sinh lại).
2. Nếu có → chạy `evaluate.py` **chỉ trên mini set đó** (không phải cả tập test), rồi
   `rescore merge` ghép kết quả vào file preds cũ.
3. `rescore rescore` — tính lại FA/CER/F1 và ghi đè report. `latency_ms`, `peak_vram_mb`
   và khối `run` được giữ nguyên từ lần chạy gốc vì chúng không phụ thuộc parser.

> ⚠ Trước khi tốn GPU: chạy cell **DEBUG 1 ẢNH — BASE MODEL** ở cuối notebook với
> `MODEL_KEY='internvl'` và nhìn chuỗi thô. Nếu nó bọc rào markdown → chạy lại rất
> đáng. Nếu là văn xuôi lan man không có JSON → con số cũ vốn đã đúng, chạy lại chỉ
> để xác nhận.

In [ ]:
# --- 5. CHẤM LẠI: lọc ảnh hỏng → sinh lại mini set → ghép → tính lại metric ---
# Chạy được nhiều lần, không hỏng thêm: lần thứ hai `missing` sẽ ra 0 ảnh và cell chỉ
# tính lại metric.
# RESCORE_PY / EVALUATE_PY đến từ cell 2c (đã dò đúng bố cục repo trên Drive).
import json
import subprocess
import sys
from pathlib import Path

assert 'RESCORE_PY' in globals(), 'Chạy cell 2c trước (nơi dò đường dẫn script).'

WORK = Path('/content/rescore')
WORK.mkdir(parents=True, exist_ok=True)

# Đặt False để chỉ tính lại từ `raw` sẵn có, TUYỆT ĐỐI không đụng GPU.
ALLOW_GPU_RERUN = True


def sh(*args):
    """Chạy một script của repo, dừng hẳn nếu nó lỗi (đừng lặng lẽ đi tiếp)."""
    cmd = [sys.executable] + [str(a) for a in args]
    print('$', ' '.join(cmd[1:]))
    if subprocess.run(cmd).returncode != 0:
        raise SystemExit(f'❌ Lỗi khi chạy: {" ".join(cmd[1:])}')


def n_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with open(path, encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())


for mode, report in [('ft', REPORT_FT), ('zeroshot', REPORT_ZEROSHOT)]:
    preds = report.with_name(report.stem + '_preds.jsonl')
    print(f'\n{"="*64}\n▶ {MODEL_KEY} / {mode} / {SIDE}\n{"="*64}')
    if not preds.exists():
        print(f'➖ Chưa có {preds.name} — chạy mục 3 trước.')
        continue

    fa_truoc = (json.loads(report.read_text(encoding='utf-8'))['field_accuracy']
                if report.exists() else None)

    # 1) Ảnh nào bắt buộc phải sinh lại bằng GPU?
    mini = WORK / f'mini_{MODEL_KEY}_{mode}_{SIDE.lower()}.jsonl'
    sh(RESCORE_PY, 'missing',
       '--preds', preds, '--test_jsonl', TEST_JSONL, '--out', mini)
    n_missing = n_lines(mini)

    # 2) Sinh lại CHỈ mini set đó, rồi ghép vào preds cũ.
    if n_missing and ALLOW_GPU_RERUN:
        tmp_report = WORK / f'tmp_{MODEL_KEY}_{mode}_{SIDE.lower()}.json'
        cmd = [EVALUATE_PY,
               '--test_jsonl', mini,
               '--base_model', MODEL_LOCAL,
               '--model_key', MODEL_KEY,
               '--image_root', IMAGE_ROOT,
               '--report_path', tmp_report]
        # Khác biệt DUY NHẤT giữa hai hàng vẫn là có gắn adapter hay không —
        # giữ đúng như mục 3 để mini set so sánh được với phần còn lại.
        cmd += ['--adapter_dir', CKPT_DIR] if mode == 'ft' else ['--zero_shot']
        sh(*cmd)
        sh(RESCORE_PY, 'merge',
           '--base', preds,
           '--patch', tmp_report.with_name(tmp_report.stem + '_preds.jsonl'),
           '--out', preds)
    elif n_missing:
        print(f'⏭  Bỏ qua {n_missing} ảnh phải sinh lại (ALLOW_GPU_RERUN=False).')

    # 3) Tính lại metric và ghi đè report.
    sh(RESCORE_PY, 'rescore', '--preds', preds, '--report', report)

    fa_sau = json.loads(report.read_text(encoding='utf-8'))['field_accuracy']
    if fa_truoc is not None:
        delta = (fa_sau - fa_truoc) * 100
        print(f'\n   FA: {fa_truoc:.2%} → {fa_sau:.2%}  ({delta:+.2f} điểm)')
        if mode == 'ft' and abs(delta) > 1e-9:
            print('   ⚠ FA của hàng FINE-TUNED lẽ ra KHÔNG được đổi (parse rate vốn '
                  'đã 100%). Dừng lại kiểm tra trước khi dùng số này.')

print('\n✅ Xong. Chạy lại cell mục 4 để xem báo cáo + Δ đã cập nhật.')

In [ ]:
# --- 5b. SOI CHUỖI THÔ CỦA NHỮNG ẢNH VẪN PARSE HỎNG ---
# Sau mục 5, những ảnh còn `pred` rỗng là các ca parser MỚI cũng chịu thua. Đọc chuỗi
# thô của chúng để phân biệt hai chuyện rất khác nhau:
#   - model trả JSON nhưng ở dạng lạ  → còn nới parser được nữa
#   - model trả văn xuôi / lặp vô hạn → đúng là model không tuân thủ schema, số hiện
#     tại phản ánh đúng năng lực zero-shot của nó
import json

N_SHOW = 3

for mode, report in [('ft', REPORT_FT), ('zeroshot', REPORT_ZEROSHOT)]:
    preds = report.with_name(report.stem + '_preds.jsonl')
    if not preds.exists():
        continue
    rows = [json.loads(line) for line in open(preds, encoding='utf-8') if line.strip()]
    bad = [r for r in rows if not r.get('pred')]
    no_raw = sum(1 for r in bad if not str(r.get('raw', '')).strip())

    print(f'\n{"="*64}\n▶ {MODEL_KEY} / {mode} / {SIDE}: '
          f'{len(bad)}/{len(rows)} ảnh vẫn parse hỏng'
          + (f'  ({no_raw} ảnh không có `raw` — file sinh trước bản vá)' if no_raw else ''))
    for r in bad[:N_SHOW]:
        raw = str(r.get('raw', '')).strip()
        print(f'\n--- {r["image"].split("/")[-1]} ---')
        print(raw[:600] + ('…' if len(raw) > 600 else '') if raw
              else '(không lưu chuỗi thô — chạy lại mục 5 với ALLOW_GPU_RERUN=True)')

In [ ]:
# --- DEBUG 1 ẢNH: FINE-TUNED ADAPTER ---
# [ĐÃ SỬA] Cell này TRƯỚC ĐÂY tự dựng BitsAndBytesConfig (thiếu double-quant) và
# gọi generate() không ép greedy → chạy bằng do_sample=True / repetition_penalty=1.05
# mặc định của Qwen. Tức là debug bằng một cấu hình, chấm điểm bằng cấu hình khác.
# Giờ dùng thẳng helper chung để KHÔNG BAO GIỜ lệch khỏi evaluate.py.
import torch, json
from PIL import Image
from transformers import AutoModelForImageTextToText
from peft import PeftModel

from src.utils.cccd_schema import SYSTEM_PROMPT
from src.models.lora_setup import build_bnb_config
from src.models.vlm_registry import (load_processor, preprocess_image,
                                     images_arg, build_gen_kwargs)

print('🔄 Đang nạp mô hình...', spec.key)
processor = load_processor(str(CKPT_DIR), spec)
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_LOCAL,
    quantization_config=build_bnb_config(torch.bfloat16),   # NF4 + double-quant, y hệt lúc train
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(base_model, str(CKPT_DIR))
model.eval()

# --- LẤY 1 ẢNH TEST ĐẦU TIÊN ---
with open(TEST_JSONL, 'r', encoding='utf-8') as f:
    rec = json.loads(f.readline().strip())

img_path = IMAGE_ROOT / rec["image"]
image = Image.open(img_path).convert("RGB")
preprocess_image(image, spec)          # hàm dùng chung train/eval/serve
human_text = rec["conversations"][0]["value"].replace("<image>", "").strip()
gold_json = rec["conversations"][1]["value"]

messages = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
    {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": human_text}]}
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=images_arg([image], spec), return_tensors="pt").to(model.device)

print("🚀 Đang chạy dự đoán ảnh:", rec["image"])
with torch.no_grad():
    generated_ids = model.generate(**inputs, **build_gen_kwargs(512))

generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]

print("\n" + "="*50)
print("🎯 ĐÁP ÁN CHUẨN (GOLD):")
print(gold_json)
print("-" * 50)
print("🤖 AI SINH RA (RAW TEXT):")
print(output_text)
print("=" * 50)

In [ ]:
# --- DEBUG 1 ẢNH: BASE MODEL (zero-shot) ---
# [ĐÃ SỬA] Cùng lý do như cell trên: dùng helper chung thay vì tự dựng
# BitsAndBytesConfig / generate() mặc định.
# ⚠ Cell này CHỈ để soi output 1 ảnh. Con số zero-shot đưa vào báo cáo phải lấy
#   từ cell `evaluate.py --zero_shot` ở trên (chạy cả test set + ghi report).
import torch, json
from PIL import Image
from transformers import AutoModelForImageTextToText

from src.utils.cccd_schema import SYSTEM_PROMPT
from src.models.lora_setup import build_bnb_config
from src.models.vlm_registry import (load_processor, preprocess_image,
                                     images_arg, build_gen_kwargs)

print("🔄 Đang nạp MÔ HÌNH GỐC (Bỏ qua LoRA)...", spec.key)
processor = load_processor(str(MODEL_LOCAL), spec)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_LOCAL,
    quantization_config=build_bnb_config(torch.bfloat16),
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()

# --- LẤY 1 ẢNH TEST ĐẦU TIÊN ---
with open(TEST_JSONL, 'r', encoding='utf-8') as f:
    rec = json.loads(f.readline().strip())

img_path = IMAGE_ROOT / rec["image"]
image = Image.open(img_path).convert("RGB")
preprocess_image(image, spec)

human_text = rec["conversations"][0]["value"].replace("<image>", "").strip()

messages = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
    {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": human_text}]}
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=images_arg([image], spec), return_tensors="pt").to(model.device)

print("🚀 Đang chạy dự đoán bằng Base Model...")
with torch.no_grad():
    generated_ids = model.generate(**inputs, **build_gen_kwargs(512))

generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]

print("🤖 BASE MODEL TRẢ LỜI:")
print(output_text)